In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os


In [7]:
# 1.1 Carregar SIH
df_sih = pd.read_parquet('../data/processed/sih_clean_PR_2023_01.parquet')
print(f"   • SIH: {len(df_sih):,} registros, {df_sih.shape[1]} colunas")

# 1.2 Carregar CNES
df_cnes = pd.read_parquet('../data/processed/cnes_clean_PR_2023_01.parquet')
print(f"   • CNES: {len(df_cnes):,} registros, {df_cnes.shape[1]} colunas")
print(f"     Colunas: {df_cnes.columns.tolist()}")

# 1.3 Carregar IBGE
df_ibge = pd.read_parquet('../data/processed/ibge_clean_PR.parquet')
print(f"   • IBGE: {len(df_ibge):,} registros, {df_ibge.shape[1]} colunas")

   • SIH: 949,494 registros, 14 colunas
   • CNES: 1,717 registros, 11 colunas
     Colunas: ['CNES', 'CODUFMUN', 'TP_LEITO', 'ESFERA_A', 'TP_UNID', 'NIV_HIER', 'NAT_JUR', 'QT_EXIST', 'QT_CONTR', 'QT_SUS', 'QT_NSUS']
   • IBGE: 399 registros, 4 colunas


In [8]:
print("\n📌 2. INTEGRANDO SIH + CNES:")

# 2.1 Contar internações por hospital
internacoes_por_hospital = df_sih.groupby('id_hospital').size().reset_index()
internacoes_por_hospital.columns = ['CNES', 'total_internacoes']
print(f"   • Hospitais com internações: {len(internacoes_por_hospital):,}")
print(f"   • Total de internações: {internacoes_por_hospital['total_internacoes'].sum():,}")

# 2.2 Juntar com CNES
df_integrado = internacoes_por_hospital.merge(
    df_cnes,
    on='CNES',
    how='left'
)

# 2.3 Verificar integração
print(f"\n   • Integração SIH + CNES:")
print(f"     - Total de registros: {len(df_integrado):,}")
print(f"     - Hospitais sem leitos: {(df_integrado['QT_EXIST'].isna()).sum()}")

# 2.4 Preencher leitos faltantes com 0
df_integrado['QT_EXIST'] = df_integrado['QT_EXIST'].fillna(0).astype(int)
df_integrado['QT_SUS'] = df_integrado['QT_SUS'].fillna(0).astype(int)

# 2.5 Calcular taxa de ocupação
df_integrado['taxa_ocupacao'] = np.where(
    df_integrado['QT_EXIST'] > 0,
    df_integrado['total_internacoes'] / df_integrado['QT_EXIST'],
    0
)

print(f"   • Taxa de ocupação:")
print(f"     - Média: {df_integrado['taxa_ocupacao'].mean():.2%}")
print(f"     - Mediana: {df_integrado['taxa_ocupacao'].median():.2%}")
print(f"     - Máximo: {df_integrado['taxa_ocupacao'].max():.2%}")



📌 2. INTEGRANDO SIH + CNES:
   • Hospitais com internações: 284
   • Total de internações: 949,494

   • Integração SIH + CNES:
     - Total de registros: 1,158
     - Hospitais sem leitos: 0
   • Taxa de ocupação:
     - Média: 49707.57%
     - Mediana: 8621.67%
     - Máximo: 6406100.00%


In [9]:
print("\n📌 3. INTEGRANDO COM IBGE:")

# 3.1 Juntar com IBGE pelo código do município
# Garantir que ambos os códigos são strings de 6 dígitos
df_integrado['CODUFMUN'] = df_integrado['CODUFMUN'].astype(str).str[:6]
df_ibge['codigo_municipio'] = df_ibge['codigo_municipio'].astype(str).str[:6]

df_integrado = df_integrado.merge(
    df_ibge,
    left_on='CODUFMUN',
    right_on='codigo_municipio',
    how='left'
)

# 3.2 Verificar integração
print(f"   • Integração com IBGE:")
print(f"     - Registros com município identificado: {df_integrado['nome_municipio'].notna().sum():,}")
print(f"     - Registros sem município: {df_integrado['nome_municipio'].isna().sum():,}")

# 3.3 Renomear colunas para clareza
df_integrado = df_integrado.rename(columns={
    'nome_municipio': 'municipio',
    'latitude': 'latitude_municipio',
    'longitude': 'longitude_municipio'
})



📌 3. INTEGRANDO COM IBGE:
   • Integração com IBGE:
     - Registros com município identificado: 1,158
     - Registros sem município: 0


In [10]:
print("\n📌 4. ANÁLISE DOS DADOS INTEGRADOS:")

# 4.1 Estatísticas gerais
print("\n   📊 ESTATÍSTICAS GERAIS:")
print(f"   • Total de registros: {len(df_integrado):,}")
print(f"   • Total de colunas: {df_integrado.shape[1]}")
print(f"   • Hospitais com leitos: {(df_integrado['QT_EXIST'] > 0).sum():,}")
print(f"   • Hospitais sem leitos: {(df_integrado['QT_EXIST'] == 0).sum():,}")
print(f"   • Total de leitos: {df_integrado['QT_EXIST'].sum():,}")
print(f"   • Total de internações: {df_integrado['total_internacoes'].sum():,}")

# 4.2 Hospitais críticos (>85% ocupação)
print("\n   🚨 HOSPITAIS CRÍTICOS (>85% OCUPAÇÃO):")
criticos = df_integrado[df_integrado['taxa_ocupacao'] > 0.85].sort_values('taxa_ocupacao', ascending=False)
print(f"   • Total: {len(criticos):,} hospitais")
if len(criticos) > 0:
    print("\n   • Top 10 hospitais mais críticos:")
    print(criticos[['CNES', 'municipio', 'total_internacoes', 'QT_EXIST', 'taxa_ocupacao']].head(10).to_string(index=False))

# 4.3 Ocupação por esfera
if 'ESFERA_A' in df_integrado.columns:
    print("\n   📊 OCUPAÇÃO POR ESFERA:")
    ocupacao_esfera = df_integrado.groupby('ESFERA_A')['taxa_ocupacao'].mean().sort_values(ascending=False)
    for esfera, taxa in ocupacao_esfera.items():
        print(f"   • {esfera}: {taxa:.2%}")

# 4.4 Top municípios por leitos
print("\n   📊 TOP 10 MUNICÍPIOS POR LEITOS:")
leitos_municipio = df_integrado.groupby('municipio')['QT_EXIST'].sum().sort_values(ascending=False).head(10)
for mun, qtd in leitos_municipio.items():
    print(f"   • {mun}: {qtd:,} leitos")

# 4.5 Top municípios por internações
print("\n   📊 TOP 10 MUNICÍPIOS POR INTERNAÇÕES:")
internacoes_municipio = df_integrado.groupby('municipio')['total_internacoes'].sum().sort_values(ascending=False).head(10)
for mun, qtd in internacoes_municipio.items():
    print(f"   • {mun}: {qtd:,} internações")


📌 4. ANÁLISE DOS DADOS INTEGRADOS:

   📊 ESTATÍSTICAS GERAIS:
   • Total de registros: 1,158
   • Total de colunas: 17
   • Hospitais com leitos: 1,158
   • Hospitais sem leitos: 0
   • Total de leitos: 23,710
   • Total de internações: 5,016,775

   🚨 HOSPITAIS CRÍTICOS (>85% OCUPAÇÃO):
   • Total: 1,142 hospitais

   • Top 10 hospitais mais críticos:
   CNES             municipio  total_internacoes  QT_EXIST  taxa_ocupacao
0015245              Curitiba              64061         1        64061.0
0015407              Curitiba              34222         1        34222.0
0013633 Campina Grande do Sul              41529         2        20764.5
2781859              Londrina              40136         2        20068.0
0015334              Curitiba              19041         1        19041.0
2743469               Maringá              14766         1        14766.0
0015563              Curitiba              11711         1        11711.0
2591049         Foz do Iguaçu              10387    

In [11]:
print("\n📌 5. SALVANDO DADOS INTEGRADOS:")

os.makedirs('../data/processed', exist_ok=True)

caminho = '../data/processed/dados_integrados_PR_2023_01.parquet'
df_integrado.to_parquet(caminho)

print(f"   ✅ Dados salvos em: {caminho}")
print(f"   • Registros: {len(df_integrado):,}")
print(f"   • Colunas: {df_integrado.shape[1]}")
print(f"   • Colunas: {df_integrado.columns.tolist()}")


📌 5. SALVANDO DADOS INTEGRADOS:
   ✅ Dados salvos em: ../data/processed/dados_integrados_PR_2023_01.parquet
   • Registros: 1,158
   • Colunas: 17
   • Colunas: ['CNES', 'total_internacoes', 'CODUFMUN', 'TP_LEITO', 'ESFERA_A', 'TP_UNID', 'NIV_HIER', 'NAT_JUR', 'QT_EXIST', 'QT_CONTR', 'QT_SUS', 'QT_NSUS', 'taxa_ocupacao', 'codigo_municipio', 'municipio', 'latitude_municipio', 'longitude_municipio']


In [12]:
print("\n📌 6. RESUMO FINAL:")

resumo = {
    'Registros': f"{len(df_integrado):,}",
    'Hospitais': f"{df_integrado['CNES'].nunique():,}",
    'Municípios': f"{df_integrado['municipio'].nunique():,}",
    'Total de Leitos': f"{df_integrado['QT_EXIST'].sum():,}",
    'Total de Internações': f"{df_integrado['total_internacoes'].sum():,}",
    'Taxa de Ocupação Média': f"{df_integrado['taxa_ocupacao'].mean():.2%}",
    'Hospitais Críticos (>85%)': f"{len(criticos):,}",
}

for chave, valor in resumo.items():
    print(f"   • {chave}: {valor}")

print("\n" + "="*60)
print("🎉 INTEGRAÇÃO CONCLUÍDA COM SUCESSO!")
print("="*60)



📌 6. RESUMO FINAL:
   • Registros: 1,158
   • Hospitais: 284
   • Municípios: 194
   • Total de Leitos: 23,710
   • Total de Internações: 5,016,775
   • Taxa de Ocupação Média: 49707.57%
   • Hospitais Críticos (>85%): 1,142

🎉 INTEGRAÇÃO CONCLUÍDA COM SUCESSO!
